In [36]:
from langchain_community.llms import Ollama
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

In [37]:
llm = Ollama(model="llama2:7b")

In [38]:
loader = CSVLoader(file_path="../data/magic_loops_data.csv")

data_csv = loader.load()

In [39]:
for document in loader.lazy_load():
    print(document)

page_content='id: 1
temperature: 73.14301741562917
timestamp: 2025-01-18 07:34:16' metadata={'source': '../data/magic_loops_data.csv', 'row': 0}
page_content='id: 1
temperature: 54.97215931693982
timestamp: 2025-01-18 07:34:16' metadata={'source': '../data/magic_loops_data.csv', 'row': 1}
page_content='id: 1
temperature: 56.9879521332875
timestamp: 2025-01-18 07:34:16' metadata={'source': '../data/magic_loops_data.csv', 'row': 2}
page_content='id: 1
temperature: 38.965496334796086
timestamp: 2025-01-18 07:34:16' metadata={'source': '../data/magic_loops_data.csv', 'row': 3}
page_content='id: 1
temperature: 57.79333662257283
timestamp: 2025-01-18 07:34:16' metadata={'source': '../data/magic_loops_data.csv', 'row': 4}
page_content='id: 1
temperature: 79.85562786783268
timestamp: 2025-01-18 07:34:16' metadata={'source': '../data/magic_loops_data.csv', 'row': 5}
page_content='id: 1
temperature: 42.429156457606744
timestamp: 2025-01-18 07:34:16' metadata={'source': '../data/magic_loops_data.

In [40]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=500)
docs = text_splitter.split_documents(data_csv)

In [41]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
embed_model = FastEmbedEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [42]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embed_model,
    persist_directory="../db/chroma_db_dir",  # Local mode with in-memory storage only
    collection_name="stanford_report_data"
)

In [43]:
vectorstore = Chroma(embedding_function=embed_model,
                     persist_directory="../db/chroma_db_dir",
                     collection_name="stanford_report_data")
retriever=vectorstore.as_retriever(search_kwargs={'k': 3})

In [44]:
custom_prompt_template = """Usa la siguiente información para responder a la pregunta del usuario.
Si no sabes la respuesta, simplemente di que no lo sabes, no intentes inventar una respuesta.

Contexto: {context}
Pregunta: {question}

Solo devuelve la respuesta útil a continuación y nada más y responde siempre en español
Respuesta útil:
"""
prompt = PromptTemplate(template=custom_prompt_template,
                        input_variables=['context', 'question'])

In [45]:
qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff",
                                 retriever=retriever,
                                 return_source_documents=True,
                                 chain_type_kwargs={"prompt": prompt})

In [46]:
response = qa.invoke({"query": "Dame el promedio de las temperaturas de la maquina del csv que te di se lo mas exacto posible"})

In [47]:
response['result']

'El promedio de las temperaturas de la máquina es de 55.2785497518547°C.'